# Connect 4 — evidence-rule benchmark

Trains **five matched arms** and rates them in a round-robin tournament.

## The four ThompsonZero cells

Search and training targets each take one of two evidence rules, independently:

| | `target = additive` | `target = additive_mle` |
|---|---|---|
| **`search = additive`** | **AA** | **AM** |
| **`search = additive_mle`** | **MA** | **MM** |

* **`additive`** — back up each leaf's *mean probability vector* and sum them, so
  a belief's concentration is **exactly the visit count** and its spread anneals
  as `1/√(n+1)`.
* **`additive_mle`** — fit a Dirichlet to the *same* observations by maximum
  likelihood (Minka's fixed point from the sufficient statistics `n, Σ log x`),
  so concentration tracks **how much the backed-up leaves actually disagree**:
  it grows while observations are scarce and saturates at the population value.

The two roles want different things from that number — search wants it to grow
so exploration anneals, targets want it to mean something the network can
predict — so the off-diagonal cells are as interesting as the diagonal ones.

## The AlphaZero control

**AZ** is the ordinary algorithm: policy head + scalar value head, PUCT
selection, visit-count policy targets, value regression to the game outcome.
Everything distributional is removed. It answers the question the four cells
can't: *does any of this beat plain AlphaZero?*

Held fixed against the ThompsonZero arms — identical trunk, identical
mover-relative observation tensor, same self-play shape (parallel games, batched
leaf evaluation, virtual loss, subtree reuse, fast/full sim caps, temperature
threshold, pool opponents), same MCTS-Solver overlay, same optimiser, LR
schedule, batch size, steps per episode and buffer size. What differs is only
the method.

The trunks are parameter-identical (59,400 each); the output layers differ
because the methods predict different things — 4 + 4×7 = 32 numbers per position
for ThompsonZero against 7 + 1 = 8 for AlphaZero. `param_counts()` prints the
split so this is checkable rather than asserted.

## Ratings

Bradley-Terry maximum likelihood over the whole result matrix, anchored at
`random = 0`. Not sequential Elo, which depends on match order and a K-factor
schedule — with a full round robin there's no reason to accept that noise.
Scores carry Wilson 95% intervals so you can see what is and isn't resolved.

In [ ]:
%pip install open_spiel -q
import os, sys, urllib.request
_BRANCH = 'claude/connect4-dirichlet-values-my96dt'
_BASE = ('https://raw.githubusercontent.com/calvinpozderac-claude/open_spiel/'
         f'{_BRANCH}/open_spiel/colabs/')
for _f in ('connect4_dirichlet_utils.py', 'connect4_alphazero_utils.py',
           'connect4_benchmark.py'):
    _p = next((p for p in (_f, os.path.join('open_spiel', 'colabs', _f),
                           os.path.join('..', 'colabs', _f))
               if os.path.exists(p)), None)
    if _p is None:
        urllib.request.urlretrieve(_BASE + _f, _f); _p = _f
    _d = os.path.dirname(os.path.abspath(_p))
    if _d not in sys.path:
        sys.path.insert(0, _d)

import importlib
import connect4_dirichlet_utils as c4
import connect4_alphazero_utils as az
import connect4_benchmark as bench
for _m in (c4, az, bench):
    importlib.reload(_m)
print('loaded', bench.__file__)

In [ ]:
# ── Everything the five arms hold in common.  Any keyword overrides a default;
# ── see bench.default_shared() for the full list.
shared = bench.default_shared(
    root         = 'c4_benchmark',   # one subdirectory per arm
    num_episodes = 4000,
    seed         = 0,
    device       = 'auto',           # 'cuda' on your GPU box

    # model (trunk is identical across engines)
    channels = 64, num_blocks = 5, head_ch = 16,

    # search / self-play
    fast_sims = 100, full_sims = 400, fast_prob = 0.75, temp_threshold = 12,
    use_workers = True, workers = 0,      # 0 = auto (cpu_count - 2)
    games_per_worker = 32, worker_wave = 8,
    pool_prob = 0.15,

    # training
    batch_size = 512, train_steps_per_ep = 8,
    lr_peak = 2e-3, lr_decay_eps = 4000,
    # The consistency term forwards each sample's SUCCESSOR too, so 1.0 nearly
    # doubles the training batch (~90% of samples have one).  Sub-sampling keeps
    # it unbiased and buys back most of that compute; drop to 0.25 if `train`
    # dominates the perf line.
    cons_frac = 1.0,

    # eval during training (the tournament below is the real measurement)
    quick_eval_every = 500, deep_eval_every = 1000, eval_sims = 64,
)

for k, v in bench.param_counts(shared).items():
    print(f'  {k:22s} {v}')
shared

In [ ]:
# Sequential on purpose: each arm already saturates the GPU through its own
# batched inference server, so running them concurrently would only distort the
# perf numbers.  Resumable — re-run the cell and each arm picks up its latest.pt.
hists = bench.train_all(shared)

In [ ]:
# Round robin 1 — SEARCH-FREE (raw network quality), both generations together
# so each arm's trajectory sits on one scale.
players = bench.load_players(shared, gens=(1000, 2000, 4000))
names, W, elo = bench.round_robin(players, sims=0, games_per_pair=60)
bench.report(names, W, elo, 'search-free · all generations')
bench.head_to_head_table(names, W)

In [ ]:
# Round robin 2 — WITH SEARCH, final checkpoints only.  This is the deployed
# configuration and the one worth the compute.
finals = bench.load_players(shared, gens=(4000,))
names_m, W_m, elo_m = bench.round_robin(finals, sims=128, games_per_pair=40)
bench.report(names_m, W_m, elo_m, 'MCTS-128 · final checkpoints')
bench.head_to_head_table(names_m, W_m)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for name, h in hists.items():
    if not h.get('ep'):
        continue
    ax[0].plot(h['ep'], h['loss'], label=name)
    ax[1].plot(h['ep'], h['draw_pct'], label=name)
    ax[2].plot(h['ep'], h['plies'], label=name)
for a, t in zip(ax, ('total loss (NOT comparable across engines)',
                     'self-play draws %', 'game length (plies)')):
    a.set_title(t); a.set_xlabel('episode'); a.legend(fontsize=8)
fig.tight_layout()

# Ratings side by side.  The loss curves above are NOT comparable between
# ThompsonZero and AlphaZero (different objectives entirely) — these are.
fig2, ax2 = plt.subplots(figsize=(8, 4.5))
order = np.argsort(-elo)
ax2.barh([names[i] for i in order][::-1], [elo[i] for i in order][::-1])
ax2.set_xlabel('Bradley-Terry Elo (random = 0)')
ax2.set_title('search-free round robin')
fig2.tight_layout();

## Self-tests

Validates the PUCT tree and its solver, the AlphaZero loss, the mixed-engine
tournament, the Bradley-Terry fit, and that the arms really are matched on every
hyperparameter they are supposed to share.

In [ ]:
import subprocess, os
_d = os.path.dirname(bench.__file__)
for _t in ('connect4_dirichlet_tests.py', 'connect4_benchmark_tests.py'):
    _r = subprocess.run([sys.executable, os.path.join(_d, _t)], cwd=_d,
                        capture_output=True, text=True)
    print(_t, '->', _r.stdout.strip().splitlines()[-1] if _r.stdout else _r.stderr[-300:])